# Anomaly Detection with anomalykit

This notebook demonstrates three anomaly detection approaches:

1. **IsolationForestDetector** - multivariate anomaly detection via random isolation
2. **MultiSensorPatternDetector** - sliding-window pattern matching across correlated sensors
3. **AdaptiveThresholdEngine** - per-asset EMA-based adaptive thresholds

We generate synthetic sensor data with injected spikes, drift, and stuck-sensor episodes, then run each detector and visualize the results.

In [ ]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from anomalykit import IsolationForestDetector, MultiSensorPatternDetector, AdaptiveThresholdEngine
from generate_data import generate_sensor_data

## Data Generation

Generate 1000 hourly readings from four correlated sensors with three anomaly types.

In [ ]:
df = generate_sensor_data(n=1000, seed=42)
sensor_cols = ["temperature", "pressure", "vibration", "flow_rate"]

print(f"Shape: {df.shape}")
print(f"Anomaly distribution:\n{df['anomaly_type'].value_counts()}")
df.head(10)

## 1. Isolation Forest Detector

In [ ]:
iso_detector = IsolationForestDetector(contamination=0.08, random_state=42)
iso_detector.fit(df[sensor_cols])
iso_result = iso_detector.detect(df[sensor_cols])

print(f"Anomalies detected: {iso_result.anomaly_mask.sum()} / {len(df)}")
print(f"Feature importance: {iso_detector.get_feature_importance()}")

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Score distribution
axes[0].hist(iso_result.scores, bins=50, color="steelblue", edgecolor="white", alpha=0.8)
axes[0].set_title("Isolation Forest - Anomaly Score Distribution")
axes[0].set_ylabel("Count")
axes[0].axvline(np.percentile(iso_result.scores, 92), color="red", ls="--", label="92nd percentile")
axes[0].legend()

# Time series with anomalies highlighted
axes[1].plot(df["timestamp"], df["temperature"], lw=0.6, color="steelblue", label="Temperature")
anomaly_idx = np.where(iso_result.anomaly_mask)[0]
axes[1].scatter(
    df["timestamp"].iloc[anomaly_idx],
    df["temperature"].iloc[anomaly_idx],
    c="red", s=12, zorder=5, label="Detected anomaly",
)
axes[1].set_title("Temperature with Detected Anomalies")
axes[1].set_ylabel("Temperature")
axes[1].legend()

# Anomaly scores over time
axes[2].fill_between(df["timestamp"], iso_result.scores, alpha=0.4, color="tomato")
axes[2].set_title("Anomaly Score Over Time")
axes[2].set_ylabel("Score")
axes[2].set_xlabel("Time")
axes[2].xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))

plt.tight_layout()
plt.show()

### Correlation Matrix Heatmap

In [ ]:
corr = df[sensor_cols].corr()

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(sensor_cols)))
ax.set_yticks(range(len(sensor_cols)))
ax.set_xticklabels(sensor_cols, rotation=45, ha="right")
ax.set_yticklabels(sensor_cols)
for i in range(len(sensor_cols)):
    for j in range(len(sensor_cols)):
        ax.text(j, i, f"{corr.values[i, j]:.2f}", ha="center", va="center", fontsize=10)
fig.colorbar(im, ax=ax, shrink=0.8)
ax.set_title("Sensor Correlation Matrix")
plt.tight_layout()
plt.show()

## 2. Multi-Sensor Pattern Detector

In [ ]:
# Split data: first 500 points as baseline, rest as test
train_df = df.iloc[:500]
test_df = df.iloc[500:]

ms_detector = MultiSensorPatternDetector(
    correlation_threshold=0.7,
    pattern_window=20,
    anomaly_threshold=2.5,
)
ms_detector.fit(train_df, sensor_cols)
ms_result = ms_detector.detect(test_df, sensor_cols)

print(f"Patterns detected: {ms_result.pattern_count}")
print(f"Anomalous points: {ms_result.anomaly_mask.sum()} / {len(test_df)}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

test_ts = test_df["timestamp"].values

for col in sensor_cols:
    axes[0].plot(test_ts, test_df[col].values, lw=0.6, label=col)

# Shade anomalous regions
mask = ms_result.anomaly_mask
axes[0].fill_between(test_ts, axes[0].get_ylim()[0], axes[0].get_ylim()[1],
                     where=mask, alpha=0.15, color="red", label="Anomaly region")
axes[0].set_title("Multi-Sensor Pattern Detection")
axes[0].legend(loc="upper right", fontsize=8)

axes[1].fill_between(test_ts, ms_result.scores, alpha=0.5, color="tomato")
axes[1].set_title("Pattern Anomaly Score")
axes[1].set_ylabel("Score")

plt.tight_layout()
plt.show()

## 3. Adaptive Threshold Engine

In [ ]:
engine = AdaptiveThresholdEngine(k_factor=3.0, ema_alpha=0.1)
at_result = engine.calculate(df, sensor_cols, asset_id="pump-01")

print(f"Total points: {at_result.total_points}")
print(f"Violations: {at_result.violation_count}")
print(f"Processing time: {at_result.processing_time_ms:.1f} ms")
print("\nBaselines:")
for bl in at_result.baselines:
    print(f"  {bl.tag_code}: mean={bl.ema_mean:.2f}, std={bl.ema_std:.2f}, "
          f"thresholds=[{bl.threshold_low:.2f}, {bl.threshold_high:.2f}]")

In [ ]:
fig, axes = plt.subplots(len(sensor_cols), 1, figsize=(14, 3 * len(sensor_cols)), sharex=True)

for i, col in enumerate(sensor_cols):
    ax = axes[i]
    bl = next((b for b in at_result.baselines if b.tag_code == col), None)
    ax.plot(df["timestamp"], df[col], lw=0.6, color="steelblue", label=col)
    if bl:
        ax.axhline(bl.threshold_high, color="red", ls="--", lw=0.8, label="Upper threshold")
        ax.axhline(bl.threshold_low, color="red", ls="--", lw=0.8, label="Lower threshold")
        ax.axhline(bl.ema_mean, color="orange", ls=":", lw=0.8, label="EMA mean")

    # Mark violations
    col_violations = [v for v in at_result.violations if v.tag_code == col]
    if col_violations:
        v_idx = [v.index for v in col_violations]
        v_val = [v.value for v in col_violations]
        ax.scatter(df["timestamp"].iloc[v_idx], v_val, c="red", s=15, zorder=5, label="Violation")

    ax.set_ylabel(col)
    ax.legend(loc="upper right", fontsize=7)

axes[-1].set_xlabel("Time")
axes[0].set_title("Adaptive Threshold Engine - Per-Sensor Thresholds")
plt.tight_layout()
plt.show()

## Summary

| Detector | Approach | Best For |
|----------|----------|----------|
| IsolationForestDetector | Random isolation of multivariate points | Point anomalies, unknown patterns |
| MultiSensorPatternDetector | Sliding-window correlation matching | Contextual anomalies, cross-sensor breakdowns |
| AdaptiveThresholdEngine | Per-tag EMA baselines | Operational monitoring, threshold alerts |